<a href="https://colab.research.google.com/github/pelineceburgun/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Lane 2 — Refresh / Content Opportunity Scoring.** Per `training-honest-models/SKILL.md`'s
question-shape table, my target (`is_declining_label = trend_direction == "down"`, defined in
W02) is a **yes/no label observed on each content item**, so I start with **Logistic
Regression** (readable baseline-of-a-model) and move to **Decision Tree** and **Random Forest**
(stronger, still inspectable at shallow depth / via feature importance). I skip Gradient
Boosting and clustering here: boosting adds complexity this label doesn't clearly need yet
(see the comparison table below), and clustering answers a different question ("what natural
groups exist") than the one I framed in W02 ("is this item worth flagging").

I also compute **permutation importance** on whichever model wins, per the skill's
"suspiciously perfect = probably leakage" warning — I do not trust an importance ranking
until I've asked whether the top feature makes sense.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`**, not a random row split. Two reasons this matters for my lane:

1. `client_id` is a pseudonym used only for grouping (per `flyrank-data/SKILL.md`) — it must
   never leak into training as a feature, but it **must** define the split, or the model can
   learn "client X's baseline traffic level" instead of the transferable pattern I actually
   want (which pages, on any client, look like they're declining).
2. In production this model would score **new clients' content**, not more pages from clients
   it already trained on — a random row split would let the model memorize per-client averages
   and overstate itself.

I hold out 20% of clients (6 of 32) entirely — none of their rows appear in training. Same
`random_state=42` as the rest of this project, so the split is reproducible.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Before touching a model — a leakage check I didn't expect to need.**
The `flyrank-data` label trap warns that `trend_direction` / `trend_pct` are never features.
I initially also included `impressions_last_30d` and `impressions_prev_30d` as features (they
seemed like ordinary engagement columns, not the label). With them in, **every model hit
precision@50 = 1.000** — a "suspiciously perfect" number the skill explicitly says to
distrust. Checking it: `trend_pct` correlates `0.9999999984` with
`(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100` — it *is* that
formula. So `impressions_last_30d` and `impressions_prev_30d` **are the label in disguise**
and I drop both from the feature set below. (`clicks_last_30d`/`prev_30d` and
`sessions_last_30d`/`prev_30d` are kept — they're correlated with the outcome but are not the
formula that defines it.)

**Metric, per W02 framing:** primary = **Precision@50** (review capacity), secondary =
average precision (whole-ranking quality) and ROC AUC (sanity check), all computed on the
**same held-out client rows**, same as the Week-4 baseline rule re-scored on this split.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

df = pd.read_csv(
    "https://raw.githubusercontent.com/pelineceburgun/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
)

# Label (W02 proxy, leakage-safe: trend_direction/trend_pct are the label family, never features)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Recreate the exact Week-4 baseline rule + score, unchanged, so it can be re-scored on this split
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_last_30d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_last_30d"]

# Grouped split by client_id (Section 2)
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = client_series.isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
print(f"clients: {len(unique_clients)} total, {n_test_clients} held out for test")
print(f"rows: {len(train_idx)} train, {len(test_idx)} test")
print(f"label rate -- train: {df['is_declining_label'].iloc[train_idx].mean():.3f}, "
      f"test: {df['is_declining_label'].iloc[test_idx].mean():.3f}")

# Feature set -- leakage-safe (impressions_last_30d / impressions_prev_30d excluded, see above)
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[categorical_features].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].reset_index(drop=True)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

def precision_at_k(y_true, scores, k=50):
    d = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    top = d.sort_values("score", ascending=False).head(min(k, len(d)))
    return float(top["y"].mean())

results = {}

# Baseline rule, re-scored on this exact test split
base_scores_test = df["baseline_score"].iloc[test_idx].to_numpy()
n_nonzero = int((base_scores_test > 0).sum())
results["baseline_rule (W4)"] = {
    "precision_at_50": precision_at_k(y_test, base_scores_test, 50),
    "avg_precision": average_precision_score(y_test, base_scores_test),
    "roc_auc": roc_auc_score(y_test, base_scores_test),
}
print(f"\nbaseline rule: only {n_nonzero} of {len(test_idx)} test rows score > 0 "
      "(the rest are 0-score ties) -- precision@50 above is unstable, see write-up below")

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = {
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "avg_precision": average_precision_score(y_test, proba),
        "roc_auc": roc_auc_score(y_test, proba),
        "precision@0.5": precision_score(y_test, pred, zero_division=0),
        "recall@0.5": recall_score(y_test, pred, zero_division=0),
        "f1@0.5": f1_score(y_test, pred, zero_division=0),
    }

results["base_rate (test rows)"] = {
    "precision_at_50": float(y_test.mean()), "avg_precision": float(y_test.mean()), "roc_auc": 0.5,
}

comparison = pd.DataFrame(results).T.round(3)
comparison

clients: 32 total, 6 held out for test
rows: 27675 train, 2325 test
label rate -- train: 0.555, test: 0.391

baseline rule: only 5 of 2325 test rows score > 0 (the rest are 0-score ties) -- precision@50 above is unstable, see write-up below


,precision_at_50,avg_precision,roc_auc,precision@0.5,recall@0.5,f1@0.5
baseline_rule (W4),0.480,0.391,0.500,NaN,NaN,NaN
logistic_regression,0.640,0.614,0.740,0.655,0.591,0.621
decision_tree,0.540,0.601,0.759,0.581,0.719,0.643
random_forest,0.860,0.670,0.775,0.576,0.755,0.653
base_rate (test rows),0.391,0.391,0.500,NaN,NaN,NaN


**Reading the table:** average precision is the fairer baseline comparison here (precision@50
for the rule is a near-coin-flip on this slice — see the "only N rows score > 0" line above:
with almost every test row tied at score 0, which rows land in the "top 50" is mostly
tie-break order, not signal. Re-running that tie-break 20 different random ways swings the
rule's precision@50 between **0.28 and 0.46**, centered almost exactly on the base rate — so
I'm reporting **average precision (0.391, indistinguishable from the base rate)** as the
trustworthy summary of the baseline's true ranking power on held-out clients, not the single
precision@50 number, which I'd be over-claiming from.

**Random Forest wins clearly** on average precision (0.670 vs. 0.391) and ROC AUC (0.775 vs.
0.500) — a real, not decorative, improvement. Logistic Regression is respectable too (0.614
avg precision) and stays fully readable (signed coefficients), which matters if this ever
needs to be explained to a non-technical reviewer. The depth-5 Decision Tree is the weakest of
the three learned models here — single-tree splits don't capture the interacting signals
(exposure × position × freshness) as well as an ensemble does, which is itself evidence for
using a forest over a single tree on this problem, not just "more complexity is better."

**Simplicity check, per the skill:** I did not try Gradient Boosting — Random Forest already
clears the baseline by a wide, stable margin, and adding a boosted model here would be
complexity the comparison hasn't earned.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Reading the errors, not just the score.**

In [3]:
# Permutation importance on the winning model (checked for "suspiciously perfect" features)
best_name = max(["logistic_regression", "decision_tree", "random_forest"],
                 key=lambda n: results[n]["precision_at_50"])
best_model = fitted[best_name]
print("best model by precision@50:", best_name)

perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
    n_jobs=-1, scoring="average_precision",
)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("\ntop 10 features (permutation importance, drop in average precision when shuffled):")
print(importance.head(10).round(4))

# Error slice: where the model's top-50 flags are wrong
test_df = df.iloc[test_idx].copy().reset_index(drop=True)
test_df["model_proba"] = best_model.predict_proba(X_test)[:, 1]
top50 = test_df.sort_values("model_proba", ascending=False).head(50)
wrong50 = top50[top50["is_declining_label"] == 0]
print(f"\ntop-50 flagged by {best_name}: {len(wrong50)} of 50 are NOT actually declining")
cols = ["content_id", "model_proba", "trend_direction", "days_since_last_update",
        "impressions_last_30d", "avg_position"]
wrong50[cols]

best model by precision@50: random_forest

top 10 features (permutation importance, drop in average precision when shuffled):
days_with_impressions    0.0652
impressions_90d          0.0418
clicks_last_30d          0.0354
sessions_last_30d        0.0223
avg_position             0.0149
sessions_prev_30d        0.0147
ctr                      0.0134
content_age_days         0.0131
position_tier_top_3      0.0128
search_volume            0.0046
dtype: float64

top-50 flagged by random_forest: 7 of 50 are NOT actually declining


,content_id,model_proba,trend_direction,days_since_last_update,impressions_last_30d,avg_position
628,content_ee6ba17be8b8,0.712094,up,20,206,17.5
1778,content_00603b0349b4,0.705004,up,20,385,25.6
2315,content_a1dd3f309e08,0.703663,up,20,2565,13.3
1968,content_b1ae59fb9582,0.700372,stable,20,473,10.6
791,content_643f585dc7f7,0.699683,up,20,238,25.1
291,content_d8c60980fcfd,0.697623,up,8,348,26.6
336,content_db1cd41b4b4f,0.696015,up,105,734,12.9


**What the model leans on:** `days_with_impressions`, `impressions_90d`, `clicks_last_30d`,
`sessions_last_30d`, and `avg_position` dominate the ranking. All five make sense for "is this
page losing ground" — persistence and volume of exposure, recent click activity, and search
rank are exactly the signals a content strategist would look at by hand. None of them is
`impressions_last_30d`/`impressions_prev_30d` (removed above) or anything position-for-position
identical to the label, so this passes the skill's sanity check this time.

**Where the model is wrong (7 of the top 50):** every miss shares a pattern — `trend_direction`
is `up` or `stable`, but `days_since_last_update` is mostly a fresh **20 days**, not stale. The
model is reading *high, sustained visibility and a good position* as decline-risk on its own,
even for pages that are currently trending up. My read: the model has learned "well-trafficked,
well-ranked pages eventually mean-revert" as a general pattern in the training clients, and
applies it even to pages that haven't started reverting yet — it's catching a directional risk
signal, not literally today's trend label. That's a real limitation to name in the paper: a
`review_for_refresh` flag from this model should be read as "worth a look," not "confirmed
declining," especially for the small share of flags that are recently updated and currently
improving — those are the ones most likely to be false alarms.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all) -- confirm after
      committing, since this was drafted and verified in a sandbox run against the same CSV
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.